In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krupalpatel07/taiwan-semiconductor-manufacturing/TSM.csv


In [2]:
# ==========================================================
# 1. IMPORTS
# ==========================================================


import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.preprocessing import PowerTransformer
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import FastICA

from IPython.display import HTML, display

pio.renderers.default = "iframe"


In [4]:
# ==========================================================
# 2. LOAD DATA
# ==========================================================

file_path = "/kaggle/input/datasets/krupalpatel07/taiwan-semiconductor-manufacturing/TSM.csv"

df = pd.read_csv(file_path)

df.columns = [c.lower() for c in df.columns]

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

df.set_index("date", inplace=True)


In [5]:
# ==========================================================
# 3. HEADER
# ==========================================================

def fab_header(title):

    display(HTML(f"""
    <div style="
    background:
    linear-gradient(
    135deg,
    #07121F,
    #0A2A43,
    #005B96,
    #00B7C2
    );
    padding:28px;
    border-radius:22px;
    margin-top:18px;
    margin-bottom:18px;
    box-shadow:0px 0px 45px rgba(0,183,194,.35);
    ">
        <h1 style="
        color:white;
        text-align:center;
        font-size:38px;
        letter-spacing:3px;
        font-family:Trebuchet MS;">
        {title}
        </h1>
    </div>
    """))

fab_header("🏭 TSM Global Fabrication Ecosystem")


In [6]:
# ==========================================================
# 4. FAB OVERVIEW
# ==========================================================

fab_header("🌍 Foundry Performance Overview")

df["returns"] = df["close"].pct_change()

annual_return = (
(
df["close"].iloc[-1]
/
df["close"].iloc[0]
)**(252/len(df))-1
)*100

annual_volatility = (
df["returns"].std()
*np.sqrt(252)
*100
)

hit_ratio = (
(df["returns"]>0).mean()
*100
)

overview = pd.DataFrame({

"Metric":[
"Annual Return %",
"Annual Volatility %",
"Positive Days %"
],

"Value":[
round(annual_return,2),
round(annual_volatility,2),
round(hit_ratio,2)
]

})

fig = px.icicle(
overview,
path=["Metric"],
values="Value",
color="Value",
title="Foundry Performance Dashboard"
)

fig.show()


In [7]:
# ==========================================================
# 5. WAFER PRODUCTION ENGINE
# ==========================================================

fab_header("⚙ Wafer Production Engine")

df["cycle10"] = df["close"].pct_change(10)

df["cycle30"] = df["close"].pct_change(30)

df["cycle90"] = df["close"].pct_change(90)

df["production_index"] = (

df["cycle10"]*0.45+

df["cycle30"]*0.35+

df["cycle90"]*0.20

)

fig = px.area(
df,
y="production_index",
title="Wafer Production Index"
)

fig.show()


In [8]:
# ==========================================================
# 6. CHIP OUTPUT PRESSURE
# ==========================================================

fab_header("📦 Chip Output Pressure")

df["output_pressure"] = (

np.sqrt(df["volume"])

*

abs(df["returns"])

)

fig = px.line(
df,
y="output_pressure",
title="Production Pressure"
)

fig.show()


In [9]:
# ==========================================================
# 7. LITHOGRAPHY ENGINE
# ==========================================================

fab_header("🔬 Lithography Precision")

df["ema18"] = df["close"].ewm(span=18).mean()

df["ema70"] = df["close"].ewm(span=70).mean()

df["precision_gap"] = (

df["ema18"]

-

df["ema70"]

)

fig = go.Figure()

fig.add_trace(

go.Scatter(

x=df.index,

y=df["precision_gap"],

fill="tozeroy",

name="Precision Gap"

)

)

fig.update_layout(
title="Lithography Precision"
)

fig.show()


In [10]:
# ==========================================================
# 8. PROCESS NODE QUALITY
# ==========================================================

fab_header("🧬 Process Node Quality")

trend = (

df["close"]

/

df["close"].rolling(120).mean()

)

volume_rank = (

df["volume"]

.rank(pct=True)

)

stability = (

1-

df["returns"]

.rolling(25)

.std()

.rank(pct=True)

)

df["node_quality"] = (

trend.rank(pct=True)*0.45+

volume_rank*0.30+

stability*0.25

)

fig = px.line(
df,
y="node_quality",
title="Node Quality Score"
)

fig.show()


In [11]:
# ==========================================================
# 9. FAB ENVIRONMENTS
# ==========================================================

fab_header("🌐 Manufacturing Environments")

momentum = df["close"].pct_change(30)

vol = df["returns"].rolling(20).std()

conditions = [

(momentum>0.15),

(momentum>0),

(momentum<0)&(vol<vol.median()),

(momentum<0)&(vol>vol.median())

]

choices = [

"3nm Expansion",

"Capacity Growth",

"Yield Optimization",

"Supply Shock"

]

df["environment"] = np.select(
conditions,
choices,
default="Balanced"
)

fig = px.scatter(
df,
x=df.index,
y="close",
color="environment",
title="Manufacturing Environment"
)

fig.show()


In [12]:
# ==========================================================
# 10. FAB CLUSTER DISCOVERY
# ==========================================================

fab_header("🏭 Foundry Cluster Discovery")

cluster = df[
[
"production_index",
"output_pressure",
"node_quality"
]
].fillna(0)

scaled = PowerTransformer().fit_transform(cluster)

model = SpectralClustering(
n_clusters=5,
random_state=42,
assign_labels="kmeans"
)

df["fab_cluster"] = model.fit_predict(scaled)

fig = px.scatter(
df,
x=df.index,
y="close",
color=df["fab_cluster"].astype(str),
title="Foundry Cluster Intelligence"
)

fig.show()


In [13]:
# ==========================================================
# 11. SILICON WAVE SEPARATION
# ==========================================================

fab_header("🌊 Silicon Wave Separation")

ica = FastICA(
n_components=2,
random_state=42
)

components = ica.fit_transform(scaled)

ica_df = pd.DataFrame(
components,
columns=["Component 1","Component 2"]
)

fig = px.scatter(
ica_df,
x="Component 1",
y="Component 2",
title="Independent Manufacturing Signals"
)

fig.show()


In [14]:
# ==========================================================
# 12. CAPACITY EXPANSION SIGNAL
# ==========================================================

fab_header("🚀 Capacity Expansion Signal")

signal = (

(df["production_index"]>

df["production_index"].rolling(40).mean())

&

(df["node_quality"]>

df["node_quality"].rolling(40).mean())

)

df["expansion"] = signal.astype(int)

fig = go.Figure()

fig.add_trace(

go.Scatter(

x=df.index,

y=df["close"],

name="TSM Price"

)

)

fig.add_trace(

go.Scatter(

x=df.index[df["expansion"]==1],

y=df["close"][df["expansion"]==1],

mode="markers",

marker=dict(size=9),

name="Expansion Zone"

)

)

fig.update_layout(
title="Capacity Expansion Opportunities"
)

fig.show()


In [15]:
# ==========================================================
# 13. FABRICATION REPORT
# ==========================================================

fab_header("📘 Foundry Intelligence Report")

print("""

1. Production Index captures multi-cycle manufacturing momentum.

2. Output Pressure measures fabrication intensity.

3. Lithography Precision monitors structural trend quality.

4. Process Node Quality evaluates stability and efficiency.

5. Manufacturing Environments classify foundry operating conditions.

6. Spectral Clustering discovers hidden production regimes.

7. FastICA isolates independent semiconductor market forces.

8. Capacity Expansion Signals identify potential long-term growth phases.

9. TSM behaves like a global semiconductor manufacturing ecosystem.

""")



1. Production Index captures multi-cycle manufacturing momentum.

2. Output Pressure measures fabrication intensity.

3. Lithography Precision monitors structural trend quality.

4. Process Node Quality evaluates stability and efficiency.

5. Manufacturing Environments classify foundry operating conditions.

6. Spectral Clustering discovers hidden production regimes.

7. FastICA isolates independent semiconductor market forces.

8. Capacity Expansion Signals identify potential long-term growth phases.

9. TSM behaves like a global semiconductor manufacturing ecosystem.


